# Lab Reto 9 — Workflow de Análisis de Devoluciones

**Sesión 9 | Módulo 4: Productionizing Data Pipelines**

## Enunciado

El equipo de operaciones necesita un **pipeline de análisis de devoluciones** que corra diariamente y produzca un reporte consolidado por tienda. El pipeline debe:

1. **Ingestar** `devoluciones_retail.csv` y `tiendas_retail.csv` a tablas Bronze
2. **Transformar** los datos a Silver: filtrar solo devoluciones aprobadas, enriquecer con nombre de tienda
3. **Agregar** en Gold: total de devoluciones y monto devuelto por tienda y motivo
4. **Orquestar** las 3 tareas como un Databricks Workflow con dependencias, retry y alerta por email

## Requisitos técnicos

- Usar `dbassociate.bronze`, `dbassociate.silver`, `dbassociate.gold`
- Parametrizar el nombre del archivo fuente con `dbutils.widgets`
- Usar Task Values para pasar el count de registros entre tareas
- Configurar el Workflow con retry (max 2) en la tarea Silver
- El Workflow debe tener una tarea de notificación final que corra **siempre** (incluso si Gold falla)

## Archivos disponibles en el Volume

```
/Volumes/dbassociate/default/vol_landing/sesion09/
  - devoluciones_retail.csv
  - tiendas_retail.csv
```

## Schema de los datos

**devoluciones_retail.csv:**
| Columna | Tipo | Descripción |
|---|---|---|
| devolucion_id | String | ID único de la devolución |
| order_id | String | ID de la orden original |
| fecha_devolucion | Date | Fecha de la solicitud |
| cliente_id | String | ID del cliente |
| motivo | String | Motivo declarado |
| monto_devuelto | Double | Monto a reembolsar |
| estado_devolucion | String | Aprobada / Rechazada / En revision |

**tiendas_retail.csv:**
| Columna | Tipo | Descripción |
|---|---|---|
| tienda_id | String | ID de la tienda (ej: T01) |
| nombre | String | Nombre de la tienda |
| ciudad | String | Ciudad |
| region | String | Región |
| gerente | String | Nombre del gerente |
| fecha_apertura | Date | Fecha de apertura |
| capacidad_m2 | Integer | Capacidad en m² |

## Output esperado en Gold

```
tienda_id | nombre_tienda | motivo | num_devoluciones | monto_total_devuelto | pct_aprobadas
T01       | Tienda Lima... | Produ... | 3              | 1150.00              | 100.0
...
```

**Runtime requerido:** DBR 13.3 LTS | **Tiempo estimado:** 60 minutos

## Paso 0 — Verificación de archivos fuente

In [0]:
# TODO: Verificar que devoluciones_retail.csv y tiendas_retail.csv existen en el Volume
# Usar dbutils.fs.ls()


## Paso 1 — Configuración de widgets y constantes

In [0]:
# TODO: Definir widgets para parametrizar el notebook
# Parámetros sugeridos: archivo de devoluciones, archivo de tiendas, fecha del batch


# TODO: Definir las constantes CATALOG, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD, VOLUME_PATH
# y los nombres de las 3 tablas que vas a crear


# Nombres de tablas sugeridos:
# bronze: dbassociate.bronze.devoluciones_raw
# bronze: dbassociate.bronze.tiendas_raw
# silver: dbassociate.silver.devoluciones_aprobadas
# gold:   dbassociate.gold.reporte_devoluciones_tienda


## Tarea 1 — Bronze: Ingesta de devoluciones y tiendas

In [0]:
# TODO: Definir el schema explícito de devoluciones_retail.csv (nunca inferSchema)
# Tipos: devolucion_id (String), order_id (String), fecha_devolucion (Date),
#        cliente_id (String), motivo (String), monto_devuelto (Double),
#        estado_devolucion (String)


# TODO: Leer devoluciones_retail.csv con el schema definido


# TODO: Agregar columnas técnicas: _ingestion_ts, _source_file, _batch_id


# TODO: Guardar como tabla Delta en dbassociate.bronze.devoluciones_raw


# TODO: Repetir el proceso para tiendas_retail.csv
# Schema: tienda_id (String), nombre (String), ciudad (String), region (String),
#         gerente (String), fecha_apertura (Date), capacidad_m2 (Integer)


# TODO: Exponer con Task Values el count de devoluciones cargadas


## Tarea 2 — Silver: Filtrar aprobadas y enriquecer con datos de tienda

In [0]:
# TODO: Leer el Task Value de la tarea anterior para validar el count
# taskKey debe coincidir con el nombre de la tarea en el Workflow


# TODO: Leer las tablas bronze de devoluciones y tiendas


# TODO: Filtrar solo devoluciones con estado_devolucion == 'Aprobada'


# TODO: Normalizar el campo motivo (trim + upper)


# TODO: Enriquecer con el nombre de tienda haciendo join con tiendas_raw
# PISTA: las devoluciones no tienen tienda_id directamente.
# Puedes asumir que order_id contiene el prefijo de tienda o usar una tabla de referencia.
# Para este lab, agrega tienda_id como columna derivada usando los primeros caracteres
# del order_id, o simplemente usa los datos de tiendas como lookup estático.


# TODO: Guardar como dbassociate.silver.devoluciones_aprobadas


# TODO: Exponer el count de registros Silver via Task Values


## Tarea 3 — Gold: Reporte de devoluciones por tienda

In [0]:
# TODO: Leer el Task Value de silver_transform


# TODO: Agregar por tienda_id y motivo:
#   - count de devoluciones (num_devoluciones)
#   - sum de monto_devuelto (monto_total_devuelto)
#   - ticket promedio de devolución
#   - fecha máxima de devolución (ultima_devolucion)


# TODO: Guardar como dbassociate.gold.reporte_devoluciones_tienda


# TODO: Mostrar el resultado final con display()


## Tarea 4 — Notificación (configurar en UI, no en código)

Esta tarea no requiere código PySpark. Se configura directamente en la UI del Workflow:

- **Task type:** Notebook (este notebook vacío, o un notebook con solo un print)
- **Depends on:** `gold_aggregation`
- **Run if:** `All done` (correr siempre, incluso si gold falla)
- El propósito es tener un punto de control final que siempre ejecute para logging o notificación

```python
# Contenido del notebook de notificación
print("Pipeline de devoluciones completado.")
print(f"Revisar tabla: dbassociate.gold.reporte_devoluciones_tienda")
```

## Configuración del Workflow en la UI

Una vez que el código funcione en modo standalone, crear el Workflow:

```
Job name: sesion09_reto_devoluciones

Tareas:
  bronze_ingest  →  silver_transform  →  gold_report  →  notify_finish
                                                          (Run if: All done)

Compute: Job cluster, DBR 13.3 LTS

Schedule: Daily at 03:00 AM

Retry en silver_transform: max_retries=2, interval=60s

Alert on failure: tu email
```

**Preguntas de reflexión:**
1. ¿Qué pasa si `bronze_ingest` falla? ¿Se ejecuta `notify_finish`?
2. ¿Cuántas veces intentaría correr `silver_transform` antes de marcar el run como fallido?
3. Si `gold_report` falla después de una ejecución exitosa de bronze y silver, ¿qué acción tomas para no re-procesar innecesariamente?

## Limpieza

In [0]:
# TODO: Agregar las sentencias DROP TABLE para limpiar las tablas creadas
# Ejecutar solo al terminar el laboratorio

# spark.sql("DROP TABLE IF EXISTS ...")
# spark.sql("DROP TABLE IF EXISTS ...")
# spark.sql("DROP TABLE IF EXISTS ...")
# spark.sql("DROP TABLE IF EXISTS ...")

print("Limpieza completada")